In [1]:
from dataclasses import dataclass

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={
        "thinking": {"type": "disabled"},
    }
)

# Agent的高级用法-流式输出

## 1、values输出模式：每次都从头输出到最新的结果

当`stream_mode`设置为values模式时，每个步骤执行后，都会输出完整的状态信息，适用于每一步都要获取完整状态、状态持久化场景。

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三","level": "VIP","join_date": "2023-01-15"}


@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费等
    """
    return {"total_orders": 15,"total_spent": 25800.00}


@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-01-31"
    }


# 创建客户服务Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions]
)

for chunk in customer_service_agent.stream(
    {
        "messages" : [
            {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="values"
):
    rprint(chunk)
    print("-" * 50)


{
    'messages': [
        HumanMessage(
            content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='69e00651-1dfd-4e71-86a2-06ce0fe701ca'
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='69e00651-1dfd-4e71-86a2-06ce0fe701ca'
        ),
        AIMessage(
            content='好的，我来查询客户 cust1234 的完整信息、历史订单和当前可用优惠活动。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 120,
                    'prompt_tokens': 455,
                    'total_tokens': 575,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 455
                },
                'model_provider': 'openai',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '805232cb-68e2-4d19-b233-aeea6ce1877a',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f3246-fa4b-7333-8735-1f03c37439fe-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_00_7w12nGLNrFhH37wiUmce0364',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_01_NIRS5FZQhISZ01cI73Rn6493',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_02_OKcjxt3UKP56nzjNKDuI8155',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 455,
                'output_tokens': 120,
                'total_tokens': 575,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='69e00651-1dfd-4e71-86a2-06ce0fe701ca'
        ),
        AIMessage(
            content='好的，我来查询客户 cust1234 的完整信息、历史订单和当前可用优惠活动。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 120,
                    'prompt_tokens': 455,
                    'total_tokens': 575,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 455
                },
                'model_provider': 'openai',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '805232cb-68e2-4d19-b233-aeea6ce1877a',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f3246-fa4b-7333-8735-1f03c37439fe-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_00_7w12nGLNrFhH37wiUmce0364',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_01_NIRS5FZQhISZ01cI73Rn6493',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_02_OKcjxt3UKP56nzjNKDuI8155',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 455,
                'output_tokens': 120,
                'total_tokens': 575,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
            name='query_customer_data',
            id='d77cd6e7-bf9f-48b5-8d20-3b68221c2926',
            tool_call_id='call_00_7w12nGLNrFhH37wiUmce0364'
        ),
        ToolMessage(
            content='{"total_orders": 15, "total_spent": 25800.0}',
            name='check_order_history',
            id='5c4f41d1-8246-461a-be88-01bd7df5c639',
            tool_call_id='call_01_NIRS5FZQhISZ01cI73Rn6493'
        ),
        ToolMessage(
            content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
            name='get_current_promotions',
            id='3fe21a45-b234-46e7-b296-6385202a2c48',
            tool_call_id='call_02_OKcjxt3UKP56nzjNKDuI8155'
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='69e00651-1dfd-4e71-86a2-06ce0fe701ca'
        ),
        AIMessage(
            content='好的，我来查询客户 cust1234 的完整信息、历史订单和当前可用优惠活动。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 120,
                    'prompt_tokens': 455,
                    'total_tokens': 575,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 455
                },
                'model_provider': 'openai',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '805232cb-68e2-4d19-b233-aeea6ce1877a',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f3246-fa4b-7333-8735-1f03c37439fe-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_00_7w12nGLNrFhH37wiUmce0364',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'cust1234'},
                    'id': 'call_01_NIRS5FZQhISZ01cI73Rn6493',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_02_OKcjxt3UKP56nzjNKDuI8155',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 455,
                'output_tokens': 120,
                'total_tokens': 575,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
            name='query_customer_data',
            id='d77cd6e7-bf9f-48b5-8d20-3b68221c2926',
            tool_call_id='call_00_7w12nGLNrFhH37wiUmce0364'
        ),
        ToolMessage(
            content='{"total_orders": 15, "total_spent": 25800.0}',
            name='check_order_history',
            id='5c4f41d1-8246-461a-be88-01bd7df5c639',
            tool_call_id='call_01_NIRS5FZQhISZ01cI73Rn6493'
        ),
        ToolMessage(
            content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
            name='get_current_promotions',
            id='3fe21a45-b234-46e7-b296-6385202a2c48',
            tool_call_id='call_02_OKcjxt3UKP56nzjNKDuI8155'
        ),
        AIMessage(
            content='以下是客户 **cust1234** 的完整信息汇总：\n\n---\n\n### 📋 客户基本信息\n| 项目 | 内容 
|\n|------|------|\n| **姓名** | 张三 |\n| **会员等级** | 🏆 **VIP** |\n| **加入日期** | 2023-01-15 |\n\n### 📦 
订单历史\n| 项目 | 数据 |\n|------|------|\n| **总订单数** | 15 单 |\n| **累计消费金额** | ¥25,800.00 |\n\n### 🎉 
当前可用优惠活动\n| 活动名称 | 有效期限 |\n|---------|---------|\n| **老用户优惠** | 至 2027-01-31 |\n| 
**会员专属折扣** | 至 2027-01-31 |\n\n---\n\n**总结：** 
客户张三（cust1234）是VIP等级的老客户，自2023年1月加入以来共完成了15笔订单，累计消费25,800元。目前可享受 
**老用户优惠** 和 **会员专属折扣** 两项促销活动，有效期至2027年1月31日。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 241,
                    'prompt_tokens': 671,
                    'total_tokens': 912,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_to

--------------------------------------------------


## 2、updates输出模式

这种模式就是默认模式。该模式中，每个步骤执行后，只增量更新状态中发生变化的内容，用于监控Agent 执行进度，例如观察Agent决定调用工具、工具执行结果等步骤。

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三","level": "VIP","join_date": "2023-01-15"}


@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费等
    """
    return {"total_orders": 15,"total_spent": 25800.00}


@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-01-31"
    }


# 创建客户服务Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions]
)

for chunk in customer_service_agent.stream(
    {
        "messages" : [
            {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="updates"
):
    rprint(chunk)
    print("-" * 50)


### 3.messages形式

In [6]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三","level": "VIP","join_date": "2023-01-15"}


@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费等
    """
    return {"total_orders": 15,"total_spent": 25800.00}


@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-01-31"
    }


# 创建客户服务Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions]
)

for chunk in customer_service_agent.stream(
    {
        "messages" : [
            {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="messages"
):
    # rprint(chunk)
    # print("-" * 50)
    print(chunk[0].content,end="",flush=True)

好的，我来为您查询客户ID为cust1234的完整信息、历史订单以及当前可用优惠活动。{"total_orders": 15, "total_spent": 25800.0}{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}以下是客户 **cust1234** 的完整信息汇总：

---

### 📋 客户基本信息
| 项目 | 内容 |
|------|------|
| **姓名** | 张三 |
| **会员等级** | 🏆 VIP |
| **加入日期** | 2023年1月15日 |

---

### 📦 历史订单概况
| 项目 | 数据 |
|------|------|
| **总订单数** | 15 单 |
| **累计消费金额** | ¥25,800.00 |

---

### 🎉 当前可用优惠活动
| 活动名称 | 有效期至 |
|---------|---------|
| 老用户优惠 | 2027年1月31日 |
| 会员专属折扣 | 2027年1月31日 |

---

**总结：** 客户张三是一位**VIP等级**的忠实老客户，自2023年1月加入以来，已累计下了 **15单**，消费总额 **¥25,800**。目前有 **两项优惠活动**（老用户优惠和会员专属折扣）可用，有效期至2027年1月31日。

### 4. tasks输出模式

该模式会输出当前task任务开始和结束的时间，包含任务的结果和错误信息，该模式用于监控任务的生命周期。

In [7]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三","level": "VIP","join_date": "2023-01-15"}


@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费等
    """
    return {"total_orders": 15,"total_spent": 25800.00}


@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-01-31"
    }


# 创建客户服务Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions]
)

for chunk in customer_service_agent.stream(
    {
        "messages" : [
            {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="tasks"
):
    rprint(chunk)
    print("-" * 50)


{
    'id': '58976674-2c07-cb9d-c244-144197173177',
    'name': 'model',
    'input': {
        'messages': [
            HumanMessage(
                content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
                additional_kwargs={},
                response_metadata={},
                id='10ea1276-529b-4305-b4a7-407ed3869de6'
            )
        ]
    },
    'triggers': ('branch:to:model',)
}

--------------------------------------------------


{
    'id': '58976674-2c07-cb9d-c244-144197173177',
    'name': 'model',
    'error': None,
    'result': {
        'messages': [
            AIMessage(
                content='好的，我来查询客户 cust1234 的完整信息、历史订单和可用优惠。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 118,
                        'prompt_tokens': 455,
                        'total_tokens': 573,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                        'prompt_cache_hit_tokens': 384,
                        'prompt_cache_miss_tokens': 71
                    },
                    'model_provider': 'openai',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                    'id': '38898082-4763-43c8-91c1-57d9a6dc2b28',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--019f3257-78e2-7881-acf9-c8ef779a8931-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_00_Gc86NnaEyCEeBgZqXMTa9454',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_01_KYydEdv6Ne6hiOLFxdAY2998',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_02_svWjlKpgwYiPdOzCUeDD7769',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 455,
                    'output_tokens': 118,
                    'total_tokens': 573,
                    'input_token_details': {'cache_read': 384},
                    'output_token_details': {}
                }
            )
        ]
    },
    'interrupts': []
}

--------------------------------------------------


{
    'id': '3cd4eee4-60a8-e837-d390-67cceb04ca44',
    'name': 'tools',
    'input': {
        '__type': 'tool_call_with_context',
        'tool_call': {
            'name': 'query_customer_data',
            'args': {'customer_id': 'cust1234'},
            'id': 'call_00_Gc86NnaEyCEeBgZqXMTa9454',
            'type': 'tool_call'
        },
        'state': {
            'messages': [
                HumanMessage(
                    content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
                    additional_kwargs={},
                    response_metadata={},
                    id='10ea1276-529b-4305-b4a7-407ed3869de6'
                ),
                AIMessage(
                    content='好的，我来查询客户 cust1234 的完整信息、历史订单和可用优惠。',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 118,
                            'prompt_tokens': 455,
                            'total_tokens': 573,
                            'completion_tokens_details': None,
                            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                            'prompt_cache_hit_tokens': 384,
                            'prompt_cache_miss_tokens': 71
                        },
                        'model_provider': 'openai',
                        'model_name': 'deepseek-v4-flash',
                        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                        'id': '38898082-4763-43c8-91c1-57d9a6dc2b28',
                        'finish_reason': 'tool_calls',
                        'logprobs': None
                    },
                    id='lc_run--019f3257-78e2-7881-acf9-c8ef779a8931-0',
                    tool_calls=[
                        {
                            'name': 'query_customer_data',
                            'args': {'customer_id': 'cust1234'},
                            'id': 'call_00_Gc86NnaEyCEeBgZqXMTa9454',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'check_order_history',
                            'args': {'customer_id': 'cust1234'},
                            'id': 'call_01_KYydEdv6Ne6hiOLFxdAY2998',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'get_current_promotions',
                            'args': {},
                            'id': 'call_02_svWjlKpgwYiPdOzCUeDD7769',
                            'type': 'tool_call'
                        }
                    ],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 455,
                        'output_tokens': 118,
                        'total_tokens': 573,
                        'input_token_details': {'cache_read': 384},
                        'output_token_details': {}
                    }
                )
            ]
        }
    },
    'triggers': ('__pregel_push',)
}

...

--------------------------------------------------


{
    'id': 'de1fa009-4514-484c-d541-11c011b888eb',
    'name': 'model',
    'input': {
        'messages': [
            HumanMessage(
                content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
                additional_kwargs={},
                response_metadata={},
                id='10ea1276-529b-4305-b4a7-407ed3869de6'
            ),
            AIMessage(
                content='好的，我来查询客户 cust1234 的完整信息、历史订单和可用优惠。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 118,
                        'prompt_tokens': 455,
                        'total_tokens': 573,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                        'prompt_cache_hit_tokens': 384,
                        'prompt_cache_miss_tokens': 71
                    },
                    'model_provider': 'openai',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                    'id': '38898082-4763-43c8-91c1-57d9a6dc2b28',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--019f3257-78e2-7881-acf9-c8ef779a8931-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_00_Gc86NnaEyCEeBgZqXMTa9454',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_01_KYydEdv6Ne6hiOLFxdAY2998',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_02_svWjlKpgwYiPdOzCUeDD7769',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 455,
                    'output_tokens': 118,
                    'total_tokens': 573,
                    'input_token_details': {'cache_read': 384},
                    'output_token_details': {}
                }
            ),
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                name='query_customer_data',
                id='c47ca7de-7c96-4b8f-bea5-9fb4fa18b180',
                tool_call_id='call_00_Gc86NnaEyCEeBgZqXMTa9454'
            ),
            ToolMessage(
                content='{"total_orders": 15, "total_spent": 25800.0}',
                name='check_order_history',
                id='e0baebff-8fca-4679-9f0b-fb9bf6b746e6',
                tool_call_id='call_01_KYydEdv6Ne6hiOLFxdAY2998'
            ),
            ToolMessage(
                content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
                name='get_current_promotions',
                id='91011fe6-a1e3-4a86-8e10-9f2af53e7ff3',
                tool_call_id='call_02_svWjlKpgwYiPdOzCUeDD7769'
            )
        ]
    },
    'triggers': ('branch:to:model',)
}

--------------------------------------------------


{
    'id': 'de1fa009-4514-484c-d541-11c011b888eb',
    'name': 'model',
    'error': None,
    'result': {
        'messages': [
            AIMessage(
                content='以下是客户 **cust1234（张三）** 的完整信息汇总：\n\n---\n\n### 📋 客户基本信息\n| 项目 | 
内容 |\n|------|------|\n| **姓名** | 张三 |\n| **客户等级** | 🏆 **VIP** |\n| **加入日期** | 2023年1月15日 
|\n\n---\n\n### 📦 历史订单情况\n| 项目 | 数据 |\n|------|------|\n| **总订单数** | 15 笔 |\n| **总消费金额** | 
**¥25,800.00** |\n\n---\n\n### 🎉 当前可用优惠活动\n| 优惠活动 | 有效期至 |\n|----------|----------|\n| 老用户优惠 
| 2027年1月31日 |\n| 会员专属折扣 | 2027年1月31日 |\n\n---\n\n**总结：** 张三作为 **VIP客户**，累计消费 
**¥25,800**（共15笔订单），目前有两项有效优惠可享（老用户优惠和会员专属折扣），有效期均至 **2027年1月31日**。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 232,
                        'prompt_tokens': 669,
                        'total_tokens': 901,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                        'prompt_cache_hit_tokens': 384,
                        'prompt_cache_miss_tokens': 285
                    },
                    'model_provider': 'openai',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                    'id': '014c8eb4-2396-48a5-bf6c-cdaea88e36d9',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f3257-80fb-7662-9a66-2d8ff9d3bebe-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 669,
                    'output_tokens': 232,
                    'total_tokens': 901,
                    'input_token_details': {'cache_read': 384},
                    'output_token_details': {}
                }
            )
        ]
    },
    'interrupts': []
}

--------------------------------------------------


## 5、debug输出模式

该模式与tasks模式类似，比task模式多输出任务步骤、时间戳、task类型（task/task_result），该模式用于调试、监控task任务的生命周期。

In [8]:
for chunk in customer_service_agent.stream(
    {
        "messages" : [
            {"role":"user","content":"查询客户id为cust1234的完整的信息、历史订单和可用优惠"}
        ]
    },
    stream_mode="debug"
):
    rprint(chunk)
    print("-" * 50)

{
    'step': 1,
    'timestamp': '2026-07-05T12:54:56.612166+00:00',
    'type': 'task',
    'payload': {
        'id': '1d064ed3-694d-be53-0054-308e634075bc',
        'name': 'model',
        'input': {
            'messages': [
                HumanMessage(
                    content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
                    additional_kwargs={},
                    response_metadata={},
                    id='f0613090-3a1e-4676-8554-00fd40622f07'
                )
            ]
        },
        'triggers': ('branch:to:model',)
    }
}

--------------------------------------------------


{
    'step': 1,
    'timestamp': '2026-07-05T12:54:58.755985+00:00',
    'type': 'task_result',
    'payload': {
        'id': '1d064ed3-694d-be53-0054-308e634075bc',
        'name': 'model',
        'error': None,
        'result': {
            'messages': [
                AIMessage(
                    content='好的，我来查询客户 cust1234 的完整信息，包括基本信息、历史订单和当前可用优惠。',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 122,
                            'prompt_tokens': 455,
                            'total_tokens': 577,
                            'completion_tokens_details': None,
                            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                            'prompt_cache_hit_tokens': 384,
                            'prompt_cache_miss_tokens': 71
                        },
                        'model_provider': 'openai',
                        'model_name': 'deepseek-v4-flash',
                        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                        'id': '61e7b266-d6fb-4439-bb15-aae08ec8b9a2',
                        'finish_reason': 'tool_calls',
                        'logprobs': None
                    },
                    id='lc_run--019f3258-9f6d-7630-bcbb-4a615a563183-0',
                    tool_calls=[
                        {
                            'name': 'query_customer_data',
                            'args': {'customer_id': 'cust1234'},
                            'id': 'call_00_dzNl0A8GQeLNsCCSLMwT5330',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'check_order_history',
                            'args': {'customer_id': 'cust1234'},
                            'id': 'call_01_RbTqrFfwT7xjYjr8NulB6575',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'get_current_promotions',
                            'args': {},
                            'id': 'call_02_amQk96sCkGwkh7ZZjkDe9905',
                            'type': 'tool_call'
                        }
                    ],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 455,
                        'output_tokens': 122,
                        'total_tokens': 577,
                        'input_token_details': {'cache_read': 384},
                        'output_token_details': {}
                    }
                )
            ]
        },
        'interrupts': []
    }
}

--------------------------------------------------


{
    'step': 2,
    'timestamp': '2026-07-05T12:54:58.756303+00:00',
    'type': 'task',
    'payload': {
        'id': '92aac84d-a274-4464-45bb-e42305749e1f',
        'name': 'tools',
        'input': {
            '__type': 'tool_call_with_context',
            'tool_call': {
                'name': 'query_customer_data',
                'args': {'customer_id': 'cust1234'},
                'id': 'call_00_dzNl0A8GQeLNsCCSLMwT5330',
                'type': 'tool_call'
            },
            'state': {
                'messages': [
                    HumanMessage(
                        content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
                        additional_kwargs={},
                        response_metadata={},
                        id='f0613090-3a1e-4676-8554-00fd40622f07'
                    ),
                    AIMessage(
                        content='好的，我来查询客户 cust1234 的完整信息，包括基本信息、历史订单和当前可用优惠。',
                        additional_kwargs={'refusal': None},
                        response_metadata={
                            'token_usage': {
                                'completion_tokens': 122,
                                'prompt_tokens': 455,
                                'total_tokens': 577,
                                'completion_tokens_details': None,
                                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                                'prompt_cache_hit_tokens': 384,
                                'prompt_cache_miss_tokens': 71
                            },
                            'model_provider': 'openai',
                            'model_name': 'deepseek-v4-flash',
                            'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                            'id': '61e7b266-d6fb-4439-bb15-aae08ec8b9a2',
                            'finish_reason': 'tool_calls',
                            'logprobs': None
                        },
                        id='lc_run--019f3258-9f6d-7630-bcbb-4a615a563183-0',
                        tool_calls=[
                            {
                                'name': 'query_customer_data',
                                'args': {'customer_id': 'cust1234'},
                                'id': 'call_00_dzNl0A8GQeLNsCCSLMwT5330',
                                'type': 'tool_call'
                            },
                            {
                                'name': 'check_order_history',
                                'args': {'customer_id': 'cust1234'},
                                'id': 'call_01_RbTqrFfwT7xjYjr8NulB6575',
                                'type': 'tool_call'
                            },
                            {
                                'name': 'get_current_promotions',
                                'args': {},
                                'id': 'call_02_amQk96sCkGwkh7ZZjkDe9905',
                                'type': 'tool_call'
                            }
                        ],
                        invalid_tool_calls=[],
                        usage_metadata={
                            'input_tokens': 455,
                            'output_tokens': 122,
                            'total_tokens': 577,
                            'input_token_details': {'cache_read': 384},
                            'output_token_details': {}
                        }
                    )
                ]
            }
        },
        'triggers': ('__pregel_push',)
    }
}

...

--------------------------------------------------


{
    'step': 3,
    'timestamp': '2026-07-05T12:54:58.799148+00:00',
    'type': 'task',
    'payload': {
        'id': '5a700bfd-ccb3-50a8-ab9a-3aefcd43567d',
        'name': 'model',
        'input': {
            'messages': [
                HumanMessage(
                    content='查询客户id为cust1234的完整的信息、历史订单和可用优惠',
                    additional_kwargs={},
                    response_metadata={},
                    id='f0613090-3a1e-4676-8554-00fd40622f07'
                ),
                AIMessage(
                    content='好的，我来查询客户 cust1234 的完整信息，包括基本信息、历史订单和当前可用优惠。',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 122,
                            'prompt_tokens': 455,
                            'total_tokens': 577,
                            'completion_tokens_details': None,
                            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                            'prompt_cache_hit_tokens': 384,
                            'prompt_cache_miss_tokens': 71
                        },
                        'model_provider': 'openai',
                        'model_name': 'deepseek-v4-flash',
                        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                        'id': '61e7b266-d6fb-4439-bb15-aae08ec8b9a2',
                        'finish_reason': 'tool_calls',
                        'logprobs': None
                    },
                    id='lc_run--019f3258-9f6d-7630-bcbb-4a615a563183-0',
                    tool_calls=[
                        {
                            'name': 'query_customer_data',
                            'args': {'customer_id': 'cust1234'},
                            'id': 'call_00_dzNl0A8GQeLNsCCSLMwT5330',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'check_order_history',
                            'args': {'customer_id': 'cust1234'},
                            'id': 'call_01_RbTqrFfwT7xjYjr8NulB6575',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'get_current_promotions',
                            'args': {},
                            'id': 'call_02_amQk96sCkGwkh7ZZjkDe9905',
                            'type': 'tool_call'
                        }
                    ],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 455,
                        'output_tokens': 122,
                        'total_tokens': 577,
                        'input_token_details': {'cache_read': 384},
                        'output_token_details': {}
                    }
                ),
                ToolMessage(
                    content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                    name='query_customer_data',
                    id='971df735-3a5b-4ccf-b4ef-f1c3c8a34178',
                    tool_call_id='call_00_dzNl0A8GQeLNsCCSLMwT5330'
                ),
                ToolMessage(
                    content='{"total_orders": 15, "total_spent": 25800.0}',
                    name='check_order_history',
                    id='bbe2aa4d-b8ca-4adb-af48-3b0e48f5b04a',
                    tool_call_id='call_01_RbTqrFfwT7xjYjr8NulB6575'
                ),
                ToolMessage(
                    content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
                    name='get_current_promotions',
                    id='f1f692f5-15a6-4f78-99bc-a9e6cc168cc2',
                    tool_call_id='call_02_amQk96sCkGwkh7ZZjkDe9905'
                )
            ]
        },
        'triggers': ('branch:to:model',)
    }
}

--------------------------------------------------


{
    'step': 3,
    'timestamp': '2026-07-05T12:55:01.802776+00:00',
    'type': 'task_result',
    'payload': {
        'id': '5a700bfd-ccb3-50a8-ab9a-3aefcd43567d',
        'name': 'model',
        'error': None,
        'result': {
            'messages': [
                AIMessage(
                    content='以下是客户 **cust1234** 的完整信息汇总：\n\n---\n\n## 🔍 客户基本信息\n| 项目 | 内容 
|\n|------|------|\n| **客户姓名** | 张三 |\n| **会员等级** | VIP |\n| **加入日期** | 2023-01-15 |\n\n---\n\n## 📦 
历史订单情况\n| 项目 | 数据 |\n|------|------|\n| **总订单数** | 15 笔 |\n| **总消费金额** | ¥25,800.00 
|\n\n---\n\n## 🎁 当前可用优惠活动\n| 优惠活动 | 有效期至 |\n|----------|---------|\n| 老用户优惠 | 2027-01-31 |\n|
会员专属折扣 | 2027-01-31 |\n\n---\n\n**总结：** 客户张三自2023年1月加入，目前是 **VIP** 等级，累计消费了 
**15笔订单**，总金额 **¥25,800**。当前可享受 **老用户优惠** 和 **会员专属折扣** 两项促销活动，有效期均至 
**2027年1月31日**。',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 239,
                            'prompt_tokens': 673,
                            'total_tokens': 912,
                            'completion_tokens_details': None,
                            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                            'prompt_cache_hit_tokens': 384,
                            'prompt_cache_miss_tokens': 289
                        },
                        'model_provider': 'openai',
                        'model_name': 'deepseek-v4-flash',
                        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                        'id': '7a12a380-54e7-4f88-80bb-abe00062440e',
                        'finish_reason': 'stop',
                        'logprobs': None
                    },
                    id='lc_run--019f3258-a7fd-7241-872e-bb6e2354e8d8-0',
                    tool_calls=[],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 673,
                        'output_tokens': 239,
                        'total_tokens': 912,
                        'input_token_details': {'cache_read': 384},
                        'output_token_details': {}
                    }
                )
            ]
        },
        'interrupts': []
    }
}

--------------------------------------------------


## 6、checkpoints输出模式

该模式中，每当检查点（checkpoint）被创建时会触发输出，输出包含检查点中的状态，用于需要状态持久化、工作流恢复或分布式执行跟踪的高级场景。

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

# 其他工具代码同上，保持不变
# ... ...
# 1. 创建内存检查点存储
checkpointer = InMemorySaver()

# 2. 创建Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions],
    checkpointer=checkpointer  # 启用检查点
)

# 3. 创建唯一的会话ID
config = {"configurable": {"thread_id": "session01"}}

# 4. 调用Agent
checkpoint_count = 0

# 使用checkpoints模式进行流式监控
for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},
        config=config,
        stream_mode="checkpoints"
):
    checkpoint_count += 1
    print(f"检查点 #{checkpoint_count}")
    print(chunk)
    print("-" * 50)

检查点 #1
{'config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'session01', 'checkpoint_id': '1f17870d-ca5e-6296-bfff-2992b1f6abe6'}}, 'parent_config': None, 'values': {'messages': []}, 'metadata': {'source': 'input', 'step': -1, 'parents': {}}, 'next': ['__start__'], 'tasks': [{'id': '27d5b136-b220-964c-6573-52058656c358', 'name': '__start__', 'interrupts': (), 'state': None}]}
--------------------------------------------------
检查点 #2
{'config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'session01', 'checkpoint_id': '1f17870d-ca62-692c-8000-78369a90e1e6'}}, 'parent_config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'session01', 'checkpoint_id': '1f17870d-ca5e-6296-bfff-2992b1f6abe6'}}, 'values': {'messages': [HumanMessage(content='查询客户ID为 CUST123456 的完整信息和可用优惠', additional_kwargs={}, response_metadata={}, id='c9a2f69d-0591-44d2-9411-e739ed76cd38')]}, 'metadata': {'source': 'loop', 'step': 0, 'parents': {}}, 'next': ['model'], 'tasks': [{'id': '166df854-06f5-

## 7、custom输出模式
## 7、custom输出模式

开发者通过`get_stream_writer`在工具或节点内部`自定义发送的数据`，用于`输出`业务逻辑相关的进度信息（如“已处理10/100条记录”）、自定义日志或指标。

In [10]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from langchain.tools import tool
import time

@tool
def generate_sales_report() -> str:
    """生成销售报告"""
    writer = get_stream_writer()

    writer({"type": "生成销售报告", "message": "开始生成销售报告"})

    # 模拟数据处理
    for i in range(1, 4):
        time.sleep(0.5)
        writer({"type": "生成销售报告","message": f"生成销售报告进度百分比：{i * 25}%"})

    writer({"type": "生成销售报告", "message": "报告生成完成"})

    return f"销售报告：总收入150万元，同比增长12%"


@tool
def generate_inventory_report() -> str:
    """生成库存报告"""
    writer = get_stream_writer()
    writer("开始库存分析...")
    time.sleep(0.5)
    writer("检查当前库存量...")
    time.sleep(0.5)
    writer("生成库存报告...")

    return "当前库存量为10000件，库存充足，无异常"


# 创建报告生成agent
reporting_agent = create_agent(
    model=model,
    tools=[generate_sales_report, generate_inventory_report]
)

for chunk in reporting_agent.stream(
        {"messages": [{"role": "user","content": "生成销售报告和库存报告"}]},
        stream_mode="custom"
):
    print(chunk)
    print("-" * 50)

{'type': '生成销售报告', 'message': '开始生成销售报告'}
--------------------------------------------------
开始库存分析...
--------------------------------------------------
检查当前库存量...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：25%'}
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：50%'}
--------------------------------------------------
生成库存报告...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：75%'}
--------------------------------------------------
{'type': '生成销售报告', 'message': '报告生成完成'}
--------------------------------------------------
